<a href="https://colab.research.google.com/github/Lee-Minsoo-97/Sales-Data-Prediction/blob/gemini_original/Valuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import drive

# --- 1. 필수 데이터 불러오기 ---
drive.mount('/content/drive', force_remount=True)

# 예측 결과 파일 불러오기
predictions_path = '/content/drive/MyDrive/Colab Notebooks/Trained Models/final_predictions.csv'
results_df = pd.read_csv(predictions_path, parse_dates=['Date'])
print("✅ 최종 예측 결과 파일을 불러왔습니다 (results_df).")

# 원본 모델링 데이터 불러오기 (원-핫 인코딩된 Brand Code 정보를 가져오기 위함)
model_data_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'
df_model = pd.read_csv(model_data_path, parse_dates=['Date'])
print("✅ 원본 모델링 데이터를 불러왔습니다 (df_model).")

# --- 2. 비즈니스 시뮬레이션: 필 레이트(Fill Rate) 계산 ---
results_df['Order_Quantity'] = results_df['Prediction_Ensemble_Weighted'] * 2
results_df['Met_Demand'] = np.minimum(results_df['Actual_Sales'], results_df['Order_Quantity'])

total_actual_sales = results_df['Actual_Sales'].sum()
total_met_demand = results_df['Met_Demand'].sum()
fill_rate = (total_met_demand / total_actual_sales) * 100

print(f"\n--- 최종 필 레이트 ---")
print(f"🚀 최종 필 레이트: {fill_rate:.2f}%")

# 재고 부족(Stock-out) 분석 (FutureWarning 해결: 0.0으로 초기화)
results_df['Stockout_Quantity'] = 0.0
stockout_mask = results_df['Actual_Sales'] > results_df['Order_Quantity']
results_df.loc[stockout_mask, 'Stockout_Quantity'] = results_df['Actual_Sales'] - results_df['Order_Quantity']

print("\n--- 재고 부족(Lost Sales)이 가장 컸던 Top 10 ---")
print(results_df.sort_values(by='Stockout_Quantity', ascending=False).head(10)[
    ['Date', 'SKU', 'Actual_Sales', 'Order_Quantity', 'Stockout_Quantity']
])


# --- ★★★ 수정된 브랜드별 필 레이트 분석 ★★★ ---

# 1. 원본 데이터에서 SKU, Date, Brand Code_* 컬럼들만 가져오기
brand_code_cols = ['SKU', 'Date'] + [col for col in df_model.columns if 'Brand Code_' in col]
brand_info_df = df_model[brand_code_cols]

# 2. 예측 결과에 원-핫 인코딩된 Brand Code 정보 통합
results_with_brand = pd.merge(results_df, brand_info_df, on=['SKU', 'Date'], how='left')

# 3. 원-핫 인코딩된 컬럼에서 원래 Brand Code 역추적
brand_cols_only = [col for col in results_with_brand.columns if 'Brand Code_' in col]
# idxmax(axis=1)는 각 행에서 값이 1인 컬럼의 이름을 찾아줌
results_with_brand['Brand_Code'] = results_with_brand[brand_cols_only].idxmax(axis=1)

# 4. Brand_Code 별로 집계하여 필 레이트 계산
brand_fill_rate_df = results_with_brand.groupby('Brand_Code').agg(
    Total_Actual_Sales=('Actual_Sales', 'sum'),
    Total_Met_Demand=('Met_Demand', 'sum')
).reset_index()

brand_fill_rate_df['Brand_Fill_Rate_%'] = 100.0
mask = brand_fill_rate_df['Total_Actual_Sales'] > 0
brand_fill_rate_df.loc[mask, 'Brand_Fill_Rate_%'] = \
    (brand_fill_rate_df['Total_Met_Demand'] / brand_fill_rate_df['Total_Actual_Sales']) * 100

# 5. 결과 출력 및 시각화
print("\n--- 브랜드 코드별 평균 필 레이트 ---")
print(brand_fill_rate_df.sort_values(by='Brand_Fill_Rate_%', ascending=False))

fig = px.bar(
    brand_fill_rate_df.sort_values(by='Brand_Fill_Rate_%', ascending=True),
    x='Brand_Fill_Rate_%',
    y='Brand_Code',  # y축을 'Brand Name' -> 'Brand_Code'로 변경
    title='브랜드 코드별 평균 필 레이트(%)',
    orientation='h',
    text=brand_fill_rate_df['Brand_Fill_Rate_%'].apply(lambda x: f'{x:.2f}%')
)
fig.update_layout(xaxis_title='Fill Rate (%)', yaxis_title='Brand Code')
fig.show()

Mounted at /content/drive
✅ 최종 예측 결과 파일을 불러왔습니다 (results_df).
✅ 원본 모델링 데이터를 불러왔습니다 (df_model).

--- 최종 필 레이트 ---
🚀 최종 필 레이트: 89.19%

--- 재고 부족(Lost Sales)이 가장 컸던 Top 10 ---
          Date       SKU  Actual_Sales  Order_Quantity  Stockout_Quantity
678 2025-07-01  NRP48814          3850         2057.68            1792.32
12  2025-07-01  APB68274          3157         1863.06            1293.94
13  2025-08-01  APB68274          2285         1730.76             554.24
17  2025-08-01  APB68276          1591         1046.84             544.16
10  2025-07-01  APB68273          1963         1670.64             292.36
157 2025-08-01  APB68515           443          223.56             219.44
161 2025-08-01  APB68517           620          449.10             170.90
335 2025-08-01  DMX48237           489          319.68             169.32
368 2025-07-01  EQQ45648           273          119.90             153.10
366 2025-07-01  EQQ45647           264          119.90             144.10

--- 브랜드 코드별 

In [ ]:
results_df

,Date,SKU,Actual_Sales,Prediction_LGBM_v1.2,Prediction_XGB_v2.0,Prediction_Ensemble_50_50,Prediction_Ensemble_Weighted,Order_Quantity,Met_Demand,Stockout_Quantity
0,2025-07-01,APB68267,949,580.64,920.19,750.42,755.42,1510.84,949.00,0.00
1,2025-08-01,APB68267,1130,360.15,682.62,521.38,526.13,1052.26,1052.26,77.74
2,2025-07-01,APB68268,1233,583.79,1011.13,797.46,803.75,1607.50,1233.00,0.00
3,2025-08-01,APB68268,1164,607.45,1046.05,826.75,833.21,1666.42,1164.00,0.00
4,2025-07-01,APB68270,435,578.00,804.40,691.20,694.53,1389.06,435.00,0.00
...,...,...,...,...,...,...,...,...,...,...
942,2025-08-01,TWA31353,8,10.63,8.53,9.58,9.55,19.10,8.00,0.00
943,2025-07-01,TWA31354,1,8.89,8.30,8.60,8.59,17.18,1.00,0.00
944,2025-08-01,TWA31354,9,10.63,8.44,9.54,9.51,19.02,9.00,0.00
945,2025-07-01,TWA60548,0,2.50,0.34,1.42,1.39,2.78,0.00,0.00


In [ ]:
# 'results_df' 데이터프레임이 있다고 가정하고 진행합니다.

# --- 1. 과재고 수량 계산 ---
# 과재고 컬럼을 0.0으로 초기화 (이전 Stockout 계산과 동일한 원리)
results_df['Overstock_Quantity'] = 0.0

# 과재고가 발생한 경우 (발주량 > 실제 판매량)
overstock_mask = results_df['Order_Quantity'] > results_df['Actual_Sales']

# 해당 경우에만 과재고 수량 계산
results_df.loc[overstock_mask, 'Overstock_Quantity'] = results_df['Order_Quantity'] - results_df['Actual_Sales']


# --- 2. 결과 분석 ---
# 총 과재고 수량 합계
total_overstock = results_df['Overstock_Quantity'].sum()

print(f"--- 최종 과재고 분석 ---")
print(f"총 발주량 합계: {results_df['Order_Quantity'].sum():,.0f}")
print(f"총 실제 판매량 합계: {results_df['Actual_Sales'].sum():,.0f}")
print(f"📦 총 과재고 수량 합계: {total_overstock:,.0f}")

print("\n--- 과재고가 가장 많이 발생한 Top 10 ---")
# 과재고 수량을 기준으로 정렬하여 상위 10개 출력
print(results_df.sort_values(by='Overstock_Quantity', ascending=False).head(10)[
    ['Date', 'SKU', 'Actual_Sales', 'Order_Quantity', 'Overstock_Quantity']
])

--- 최종 과재고 분석 ---
총 발주량 합계: 103,271
총 실제 판매량 합계: 73,292
📦 총 과재고 수량 합계: 37,902

--- 과재고가 가장 많이 발생한 Top 10 ---
          Date       SKU  Actual_Sales  Order_Quantity  Overstock_Quantity
438 2025-07-01  HBF94449           102         1084.10              982.10
4   2025-07-01  APB68270           435         1389.06              954.06
522 2025-07-01  LFT68101           369         1208.88              839.88
265 2025-08-01  DMX48141           304          929.94              625.94
5   2025-08-01  APB68270           337          941.74              604.74
526 2025-07-01  LFT68201           603         1200.24              597.24
0   2025-07-01  APB68267           949         1510.84              561.84
334 2025-07-01  DMX48237            79          637.14              558.14
520 2025-07-01  LFT67921           626         1170.62              544.62
8   2025-07-01  APB68272           417          929.80              512.80


In [ ]:
results_df

,Date,SKU,Actual_Sales,Prediction_LGBM_v1.2,Prediction_XGB_v2.0,Prediction_Ensemble_50_50,Prediction_Ensemble_Weighted,Order_Quantity,Met_Demand,Stockout_Quantity,Overstock_Quantity
0,2025-07-01,APB68267,949,580.64,920.19,750.42,755.42,1510.84,949.00,0.00,561.84
1,2025-08-01,APB68267,1130,360.15,682.62,521.38,526.13,1052.26,1052.26,77.74,0.00
2,2025-07-01,APB68268,1233,583.79,1011.13,797.46,803.75,1607.50,1233.00,0.00,374.50
3,2025-08-01,APB68268,1164,607.45,1046.05,826.75,833.21,1666.42,1164.00,0.00,502.42
4,2025-07-01,APB68270,435,578.00,804.40,691.20,694.53,1389.06,435.00,0.00,954.06
...,...,...,...,...,...,...,...,...,...,...,...
942,2025-08-01,TWA31353,8,10.63,8.53,9.58,9.55,19.10,8.00,0.00,11.10
943,2025-07-01,TWA31354,1,8.89,8.30,8.60,8.59,17.18,1.00,0.00,16.18
944,2025-08-01,TWA31354,9,10.63,8.44,9.54,9.51,19.02,9.00,0.00,10.02
945,2025-07-01,TWA60548,0,2.50,0.34,1.42,1.39,2.78,0.00,0.00,2.78


In [ ]:
import plotly.express as px

# 'results_df'에 'Overstock_Quantity' 컬럼이 있다고 가정합니다.

# 과재고가 0보다 큰 제품들만 선택
overstock_df = results_df[results_df['Overstock_Quantity'] > 0]

# 과재고 수량을 기준으로 상위 15개 정렬
top_overstock = overstock_df.sort_values(by='Overstock_Quantity', ascending=False).head(15)

# 막대그래프 생성
fig = px.bar(
    top_overstock,
    x='SKU',
    y='Overstock_Quantity',
    title='과재고가 가장 많이 발생한 Top 15 SKU',
    labels={'Overstock_Quantity': '과재고 수량', 'SKU': 'SKU 번호'},
    text=top_overstock['Overstock_Quantity'].apply(lambda x: f'{x:,.0f}') # 막대 위에 수치 표시
)

fig.show()

## **판매량 예측 프로젝트 최종 요약 및 성과**

### **A. 프로젝트 목표**
월별 판매 데이터와 PO(Purchase Order) 데이터를 통합하여, SKU별 월간 판매량을 예측하는 고성능 머신러닝 모델을 구축하고, 그 결과를 비즈니스 관점에서 심층 분석한다.

---
### **B. 수행 과정 요약**
1.  **데이터 통합:** 분산된 `Sales` CSV 파일들과 단일 `PO` CSV 파일을 병합하여, `Sales`와 `PO_Quantity`를 모두 포함하는 월별/SKU별 마스터 데이터프레임을 생성했다.
2.  **피처 엔지니어링:** 모델의 예측력을 극대화하기 위해 다음과 같은 핵심 피처들을 생성했다.
    * **시간 기반 피처:** 연도, 월, 분기 등 주기성 학습
    * **시차 피처 (Lag Features):** 과거 1~3개월의 판매량과 PO 수량으로 과거 추세 반영
    * **이동 평균 피처 (Rolling Features):** 최근 3개월 평균으로 단기 트렌드 포착
    * **범주형 피처:** `Brand Code`, `Status`를 원-핫 인코딩으로 변환
3.  **v1.x 모델링 (LightGBM):**
    * **v1.0:** 기본 LightGBM 모델을 구축 (MAE 37.87).
    * **v1.1:** 시계열 교차 검증과 `Optuna`를 활용한 하이퍼파라미터 튜닝으로 모델을 견고하게 개선 (CV MAE 28.67, 최종 Test MAE 32.13).
4.  **v2.x 모델링 (XGBoost & Ensemble):**
    * **v2.0:** GPU 가속을 활용하여 XGBoost 모델을 동일한 방식으로 튜닝 (CV MAE 27.03), LightGBM보다 나은 잠재력 확인.
    * **v2.1:** 최종적으로 v1.1(LightGBM)과 v2.0(XGBoost) 모델을 **앙상블**하여, 두 모델의 장점을 결합한 최종 예측 모델을 완성했다.
5.  **심층 분석:** 최종 앙상블 모델의 예측 결과를 바탕으로 MAE/MAPE, Top 10 오차, 오차 분포 시각화, 비즈니스 규칙을 적용한 **필 레이트(Fill Rate)**, **재고 부족(Stockout)**, **과재고(Overstock)** 분석을 수행하여 모델의 강점과 약점을 명확히 진단했다.

---
### **C. 최종 결과물 및 핵심 인사이트**
1.  **예측 모델 자산:**
    * 최적화된 LightGBM 모델 (`lgbm_model_v1.2.pkl`)
    * 최적화된 XGBoost 모델 (`xgb_model_v2_0.pkl`)
    * 두 모델을 결합한 최종 앙상블(v2.1) 예측 로직
2.  **핵심 인사이트:**
    * 우리 모델은 판매량이 안정적인 **대다수의 '롱테일' 제품군에 대해 매우 정확한 예측**을 수행함을 확인했다.
    * 반면, 판매량이 폭발적이거나 변동성이 큰 소수의 **'슈퍼스타' 제품군 예측에는 어려움**을 겪으며, 이는 현재 데이터에 없는 **'프로모션'과 같은 외부 요인**이 가장 큰 원인임을 강하게 시사한다.
    * 현재의 예측 모델과 발주 전략(예측x2) 적용 시, 약 **89%의 고객 수요를 만족**시킬 수 있으며, 어떤 제품에서 재고 부족과 과재고가 주로 발생하는지 구체적으로 식별했다.

---
### **D. 향후 전략 제언**
* **단기 전략 (즉시 실행):** 완성된 앙상블 모델을 활용하여, **'롱테일' 제품군의 수요 예측 및 발주를 자동화**하고, **'핵심' 제품군은 모델 예측을 참고 자료로 활용**하여 담당자가 최종 검토하는 하이브리드 전략을 도입한다.
* **장기 전략 (v3.0):** 모델 성능의 다음 단계 도약을 위해, **'프로모션 데이터(할인율 등)'**를 수집하여 새로운 피처로 추가하는 데이터 강화 프로젝트를 시작하는 것을 최우선으로 권장한다.